# IzzyViz Use Case 1: Exploring RAG Attention Behavior

This notebook demonstrates the IzzyViz workflow on one RAG-style extractive QA example.

Workflow:

**Dataset Overview -> Attention Tensors -> Overview -> Select -> Cluster -> Inspect -> Explore**

Research focus:

- The base SQuAD-style context, question, and gold answer stay fixed.
- We add one retrieved context that contains the answer phrase and repeated answer-related evidence.
- We use IzzyViz to inspect whether attention heads focus on answer-bearing spans, repeated evidence, or structural tokens such as prompt prefixes and sinks.

The notebook generates the selected RAG example from the included dataset. It does not rely on pre-rendered local experiment images.

## 0. Clone and Install IzzyViz

This use case can be opened directly from GitHub/Colab. The setup cell below finds an existing local `IzzyViz` checkout or clones `https://github.com/zoeyada/IzzyViz.git`.

In local VS Code runs, the notebook uses the checkout directly on `sys.path`. In Colab, it installs IzzyViz in editable mode. The workflow itself computes attention tensors in memory, then runs Overview, Select, Cluster, and Inspect with the same API pattern as the quick-start workflow notebook.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/zoeyada/IzzyViz.git'
REPO_NAME = 'IzzyViz'

def find_or_clone_project_dir():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'izzyviz').is_dir() and (candidate / 'workflow').is_dir():
            return candidate

    workspace = Path('/content') if Path('/content').exists() else cwd
    repo_dir = workspace / REPO_NAME
    if not repo_dir.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(repo_dir)])
    return repo_dir.resolve()

PROJECT_DIR = find_or_clone_project_dir()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if Path('/content').exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR)])
else:
    print('Using local checkout on sys.path; skipping editable install.')

def ensure_numpy_pandas_abi():
    check_code = "import numpy, pandas; import numpy.random; print(numpy.__version__, pandas.__version__)"
    result = subprocess.run([sys.executable, '-c', check_code], text=True, capture_output=True)
    if result.returncode == 0:
        print('NumPy/Pandas:', result.stdout.strip())
        return

    print('Repairing NumPy/Pandas binary install:')
    print((result.stderr or result.stdout).strip().splitlines()[-1])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--force-reinstall',
        '--no-cache-dir',
        'numpy==1.26.4',
        'pandas==2.2.2',
    ])

ensure_numpy_pandas_abi()
print('Using IzzyViz project:', PROJECT_DIR)


## 1. Setup and Dataset Overview

This first overview is only about the RAG dataset content. It does not use model attention, clustering, or any computed visualization result.

Key parameters:

- `TARGET_CTX_ID`: the retrieved context to run through the full IzzyViz workflow.
- `ANSWER`: the gold SQuAD answer used as the analysis anchor.
- `MODEL_NAME`: the causal LM used to compute attention tensors for this RAG prompt.


In [ ]:
from pathlib import Path
import gc
import json
import os
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Image, IFrame, Markdown
from transformers import AutoModelForCausalLM, AutoTokenizer

from izzyviz import (
    visualize_attention_overview_fast,
    select_attention_heads_by_metric,
    cluster_attention_heads,
    visualize_attention_matrix,
)

PROJECT_DIR = Path(PROJECT_DIR) if 'PROJECT_DIR' in globals() else Path.cwd()
if PROJECT_DIR.name != 'IzzyViz' and (PROJECT_DIR.parent / 'izzyviz').exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATASET_DIR = PROJECT_DIR / 'workflow' / 'rag_cluster' / 'rag_contexts_answer_bearing'
OUTPUT_ROOT = DATASET_DIR / 'exp_function'
CONTEXTS_PATH = DATASET_DIR / 'external_contexts.json'

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
TARGET_CTX_ID = 'ctx_ac_005'
ANSWER = 'Lobund Institute for Animal Studies'
ANSWER_WORDS = ['lobund', 'institute', 'animal', 'studies']
SAVE_DATA_OUTPUTS = True
HF_LOCAL_FILES_ONLY = False

if not CONTEXTS_PATH.exists():
    raise FileNotFoundError(f'Missing RAG dataset: {CONTEXTS_PATH}')

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def count_answer_word_hits(text):
    text_l = text.lower()
    return sum(text_l.count(word) for word in ANSWER_WORDS)

def safe_name(value):
    import re
    value = str(value)
    value = re.sub(r'[^a-zA-Z0-9_\-]+', '_', value)
    return value.strip('_') or 'NA'

def show_image(path, width=1100):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path), width=width))

contexts = load_json(CONTEXTS_PATH)
if TARGET_CTX_ID not in contexts:
    raise KeyError(f'{TARGET_CTX_ID} was not found in {CONTEXTS_PATH}')

dataset_rows = []
for ctx_id, item in contexts.items():
    answer_overlap = item.get('answer_overlap') or {}
    text = item.get('text', '')
    dataset_rows.append({
        'ctx_id': ctx_id,
        'length': int(item.get('length', -1)),
        'relatedness': item.get('relatedness', 'unknown'),
        'full_answer_present': bool(answer_overlap.get('full_answer_present', False)),
        'answer_overlap_ratio': float(answer_overlap.get('overlap_ratio', 0.0) or 0.0),
        'answer_word_hits': count_answer_word_hits(text),
        'text_preview': text[:260].replace('\n', ' '),
    })

dataset_df = pd.DataFrame(dataset_rows).sort_values(['relatedness', 'full_answer_present', 'length', 'ctx_id']).reset_index(drop=True)
target_item = contexts[TARGET_CTX_ID]
target_answer_overlap = target_item.get('answer_overlap') or {}

OUTPUT_DIR = OUTPUT_ROOT / f"{TARGET_CTX_ID}_{target_item.get('length', 'NA')}_{safe_name(target_item.get('relatedness', 'NA'))}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_DIR)
print('Dataset:', DATASET_DIR)
print('Output:', OUTPUT_DIR)
print('Number of retrieved contexts:', len(dataset_df))
print('Target context:', TARGET_CTX_ID)
print('Target length:', target_item.get('length'))
print('Target relatedness:', target_item.get('relatedness'))
print('Target contains full answer:', target_answer_overlap.get('full_answer_present'))
print('Target answer word hits:', count_answer_word_hits(target_item.get('text', '')))

display(dataset_df[['ctx_id', 'relatedness', 'length', 'full_answer_present', 'answer_overlap_ratio', 'answer_word_hits']].head(12))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

related_counts = dataset_df['relatedness'].value_counts().sort_index()
axes[0].bar(related_counts.index.astype(str), related_counts.values, color='#7b3fb2')
axes[0].set_title('Retrieved Contexts by Relatedness')
axes[0].set_xlabel('Relatedness')
axes[0].set_ylabel('Count')

axes[1].hist(dataset_df['length'], bins=12, color='#9b6bd3', edgecolor='white')
axes[1].axvline(target_item.get('length'), color='#4b146f', linestyle='--', linewidth=2, label=TARGET_CTX_ID)
axes[1].set_title('Retrieved Context Lengths')
axes[1].set_xlabel('Length')
axes[1].set_ylabel('Count')
axes[1].legend()

answer_counts = dataset_df['full_answer_present'].value_counts().reindex([False, True], fill_value=0)
axes[2].bar(['No full answer', 'Full answer'], answer_counts.values, color=['#c9b3e6', '#6f2da8'])
axes[2].set_title('Answer-Bearing Contexts')
axes[2].set_ylabel('Count')

plt.show()

print('Target retrieved context preview:')
print(target_item.get('text', '')[:900])

## 2. Attention Tensors

This section follows the quick-start workflow notebook: build the RAG prompt, run one forward pass, keep `attentions` in memory, and define the `x` and `y` slices used by every visualization stage.


In [ ]:
BASE_CONTEXT = (
    "The Rev. John J. Cavanaugh, C.S.C. served as president from 1946 to 1952. "
    "Cavanaugh's legacy at Notre Dame in the post-war years was devoted to raising academic standards "
    "and reshaping the university administration to suit it to an enlarged educational mission and an expanded "
    "student body and stressing advanced studies and research at a time when Notre Dame quadrupled in student census, "
    "undergraduate enrollment increased by more than half, and graduate student enrollment grew fivefold. "
    "Cavanaugh also established the Lobund Institute for Animal Studies and Notre Dame's Medieval Institute. "
    "Cavanaugh also presided over the construction of the Nieuwland Science Hall, Fisher Hall, and the Morris Inn, "
    "as well as the Hall of Liberal Arts (now O'Shaughnessy Hall), made possible by a donation from I.A. O'Shaughnessy, "
    "at the time the largest ever made to an American Catholic university. "
    "Cavanaugh also established a system of advisory councils at the university, which continue today and are vital "
    "to the university's governance and development."
)
QUESTION = 'Which institute involving animal life did Cavanaugh create at Notre Dame?'

rag_context = target_item['text']
rag_context_text = f'RAG Context: {rag_context}\n'
base_context_text = f'Context: {BASE_CONTEXT}\n'
question_text = f'Question: {QUESTION}\n'
answer_prefix = 'Answer:'
answer_text = f' {ANSWER}'

x_text = rag_context_text + base_context_text + question_text
prompt = x_text + answer_prefix + answer_text

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

print('Loading tokenizer:', MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False,
    local_files_only=HF_LOCAL_FILES_ONLY,
)

print('Loading model:', MODEL_NAME)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
    attn_implementation='eager',
    trust_remote_code=False,
    local_files_only=HF_LOCAL_FILES_ONLY,
)
model.to(DEVICE)
model.eval()
print('Loaded in', round(time.time() - t0, 1), 's')
print('Device:', DEVICE)
print('Model dtype:', next(model.parameters()).dtype)

inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(DEVICE)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

x_end = len(tokenizer(x_text, add_special_tokens=False)['input_ids'])
answer_start = len(tokenizer(x_text + answer_prefix, add_special_tokens=False)['input_ids'])
answer_end = len(tokens)

print('Sequence length:', len(tokens))
print('x span = RAG context + base context + question:', (0, x_end))
print('y span = answer:', (answer_start, answer_end))
print('Answer tokens:', tokens[answer_start:answer_end])

print('Running model forward pass...')
t0 = time.time()
with torch.no_grad():
    outputs = model(
        **inputs,
        output_attentions=True,
        return_dict=True,
        use_cache=False,
    )
print('Forward done in', round(time.time() - t0, 1), 's')

attentions = outputs.attentions
num_layers = len(attentions)
num_heads = attentions[0].shape[1]

print('Layers:', num_layers)
print('Heads per layer:', num_heads)
print('Layer 0 attention shape:', tuple(attentions[0].shape))

attention_region = np.stack([
    layer_attn[0, :, answer_start:answer_end, 0:x_end].detach().float().cpu().numpy()
    for layer_attn in attentions
])
x_labels = tokens[:x_end]
y_labels = tokens[answer_start:answer_end]
print('answer -> RAG/context/question region:', attention_region.shape)


## 3. Overview

Show the full Layer x Head overview twice, matching the workflow notebook: first without virtual-token merging, then with merged virtual tokens.


In [ ]:
overview_no_merge_path = OUTPUT_DIR / '01_overview_answer_to_rag_context_question.png'
fig, axes = visualize_attention_overview_fast(
    attentions,
    query_slice=(answer_start, answer_end),
    key_slice=(0, x_end),
    title='RAG Overview: answer -> RAG context/base context/question attention',
    save_path=overview_no_merge_path,
    shared_color_scale=True,
    shared_cbar=True,
    shared_cbar_label='Attention score',
    max_display_cols=96,
    close_after_save=True,
)
show_image(overview_no_merge_path)

overview_merge_path = OUTPUT_DIR / '01_overview_answer_to_rag_context_question_merge_tokens.png'
fig, axes = visualize_attention_overview_fast(
    attentions,
    query_slice=(answer_start, answer_end),
    key_slice=(0, x_end),
    title='RAG Overview with Virtual Tokens: answer -> RAG context/base context/question attention',
    save_path=overview_merge_path,
    shared_color_scale=True,
    shared_cbar=True,
    shared_cbar_label='Attention score',
    merge_virtual_tokens=True,
    overview_top_n=3,
    show_merge_token_labels=True,
    merge_token_label_mode='index',
    merge_token_important_label_mode='index',
    merge_token_label_fontsize=3,
    x_labels=x_labels,
    y_labels=y_labels,
    close_after_save=True,
)
show_image(overview_merge_path)


## 4. Select

### 4.1 Metric-Based Head Ranking

Rank heads by a selectable metric. The notebook displays the metric-score heatmap and the detailed top-k attention maps, matching the quick-start workflow.


In [ ]:
METRIC_NAME = 'long_range'
METRIC_PARAMS = {'min_distance': 20}
TOP_K = 8

selection_dir = OUTPUT_DIR / f'02_select_{METRIC_NAME}'
selection_result = select_attention_heads_by_metric(
    attentions,
    query_slice=(answer_start, answer_end),
    key_slice=(0, x_end),
    tokens=tokens,
    metric_name=METRIC_NAME,
    metric_params=METRIC_PARAMS,
    ignore_first_n=1,
    top_k=TOP_K,
    output_dir=selection_dir,
    run_name=f'{TARGET_CTX_ID}_{METRIC_NAME}_answer_to_rag_x',
    plot_overview=True,
    plot_overview_no_merge=True,
    plot_overview_merge=True,
    plot_importance_heatmap=True,
    plot_top_attention_maps=True,
    top_attention_count=TOP_K,
    overview_renderer='fast',
    overview_top_n=3,
    overview_kwargs={
        'figsize': (30, 50),
        'shared_cbar': True,
        'shared_cbar_label': 'Attention score',
        'highlight_top_cells': True,
        'highlight_ranked_only_no_merge': True,
        'highlight_ranked_only_merge': False,
        'show_merge_token_labels': True,
        'merge_token_label_mode': 'index',
        'merge_token_important_label_mode': 'index',
        'merge_token_label_fontsize': 3,
        'wspace': 0.24,
        'hspace': 0.26,
    },
    detail_top_n=20,
    detail_merge_virtual_tokens=True,
    detail_file_format='png',
    detail_kwargs={
        'xlabel': 'x: RAG Context + Base Context + Question',
        'ylabel': 'y: Answer',
        'length_threshold': 64,
        'if_interval': False,
        'if_top_cells': True,
        'show_scores_in_enlarged_cells': False,
        'lean_more': True,
    },
    close_after_save=True,
    save_summary=SAVE_DATA_OUTPUTS,
)

selection_df = pd.DataFrame(selection_result['top_records'])
display(selection_df)
for record in selection_result['top_records']:
    print(f"rank {record['rank']:02d}: L{record['layer']:02d} H{record['head']:02d}, score={record['score']:.6f}")

show_image(selection_result['output_paths']['overview_no_merge'])
show_image(selection_result['output_paths']['overview_merge'])
show_image(selection_result['output_paths']['importance_heatmap'], width=900)

for detail_path in selection_result['output_paths'].get('top_attention_maps', [])[:TOP_K]:
    show_image(detail_path, width=1000)


### 4.2 Clustering-Based Representative Heads

Cluster heads by multiple attention features, then show PCA, both overview variants, and the representative heads that have visually distinctive behavior.


In [ ]:
cluster_dir = OUTPUT_DIR / '03_cluster_representative_heads'
cluster_result = cluster_attention_heads(
    attentions,
    query_slice=(answer_start, answer_end),
    key_slice=(0, x_end),
    tokens=tokens,
    metrics=[
        'concentration',
        'entropy',
        'variance',
        'threshold_mass',
        'top_percent_mass',
        'low_range',
        'long_range',
    ],
    metric_params={
        'threshold_mass': {'threshold': 0.05},
        'top_percent_mass': {'top_percent': 0.05},
        'low_range': {'max_distance': 5},
        'long_range': {'min_distance': 20},
    },
    n_clusters=5,
    representative_per_cluster=1,
    ignore_first_n=1,
    max_detail_plots=5,
    output_dir=cluster_dir,
    run_name=f'{TARGET_CTX_ID}_answer_to_rag_x_cluster',
    overview_renderer='fast',
    plot_overview=True,
    plot_overview_no_merge=True,
    plot_overview_merge=True,
    plot_pca=True,
    plot_detail_heatmaps=True,
    overview_top_n=5,
    overview_kwargs={
        'figsize': (30, 50),
        'shared_cbar': True,
        'shared_cbar_label': 'Attention score',
        'show_merge_token_labels': True,
        'merge_token_labels_representatives_only': False,
        'merge_token_label_mode': 'index',
        'merge_token_important_label_mode': 'index',
        'merge_token_label_fontsize': 3,
        'wspace': 0.24,
        'hspace': 0.26,
    },
    detail_top_n=20,
    detail_merge_virtual_tokens=True,
    save_summary=SAVE_DATA_OUTPUTS,
    detail_kwargs={
        'xlabel': 'x: RAG Context + Base Context + Question',
        'ylabel': 'y: Answer',
        'length_threshold': 64,
        'if_interval': False,
        'if_top_cells': True,
        'show_scores_in_enlarged_cells': False,
        'lean_more': True,
    },
)

cluster_sizes = pd.Series(cluster_result['labels']).value_counts().sort_index().to_dict()
representative_records = []
for idx in cluster_result['representatives']:
    layer, head = cluster_result['head_infos'][idx]
    cluster_id = int(cluster_result['labels'][idx])
    representative_records.append({
        'cluster': cluster_id,
        'layer': int(layer),
        'head': int(head),
        'cluster_size': int(cluster_sizes.get(cluster_id, 0)),
    })
    print(f'cluster {cluster_id}: L{layer:02d} H{head:02d}')

representative_df = pd.DataFrame(representative_records).sort_values('cluster').reset_index(drop=True)
display(representative_df)

show_image(cluster_result['output_paths']['pca'], width=850)
show_image(cluster_result['output_paths']['overview_no_merge'])
show_image(cluster_result['output_paths']['overview_merge'])

display(Markdown('### Characteristic Representative Head Heatmaps'))
for detail_path in cluster_result['output_paths'].get('detail_heatmaps', []):
    detail_path = Path(detail_path)
    display(Markdown(f'**{detail_path.name}**'))
    display(IFrame(src=str(detail_path), width=920, height=560))


## 5. Inspect

Inspect one concrete Layer/Head with `visualize_attention_matrix`. By default this uses the first cluster representative; change `USER_LAYER` and `USER_HEAD` to inspect another selected head.


In [ ]:
if representative_records:
    USER_LAYER = representative_records[0]['layer']
    USER_HEAD = representative_records[0]['head']
else:
    USER_LAYER = selection_result['top_records'][0]['layer']
    USER_HEAD = selection_result['top_records'][0]['head']

print('Showing L', USER_LAYER, 'H', USER_HEAD)
user_matrix = attention_region[USER_LAYER, USER_HEAD]

inspect_path = OUTPUT_DIR / f'04_inspect_L{USER_LAYER}_H{USER_HEAD}_virtual_tokens.png'
ax, _ = visualize_attention_matrix(
    user_matrix,
    x_labels=x_labels,
    y_labels=y_labels,
    title=f'Inspect RAG Attention: {TARGET_CTX_ID} L{USER_LAYER} H{USER_HEAD}',
    xlabel='x: RAG Context + Base Context + Question',
    ylabel='y: Answer',
    top_n=20,
    enlarged_size=2.0,
    gamma=1.5,
    cmap='Purples',
    merge_virtual_tokens=True,
    virtual_token_min_run=2,
    length_threshold=64,
    if_interval=False,
    if_top_cells=True,
    show_scores_in_enlarged_cells=False,
    lean_more=True,
    save_path=inspect_path,
    close_after_save=True,
)
show_image(inspect_path, width=1050)


## 6. Explore: RAG-Specific Questions

This final section is for interpretation rather than another computation block.

For the demo paper, this use case can support questions such as:

- In extractive QA, do any heads clearly attend to answer-bearing tokens or nearby context?
- When retrieved evidence repeats answer-related content, does attention split across repeated spans or concentrate on one occurrence?
- Are representative heads driven by answer evidence, long-range retrieval structure, or persistent attention sinks?
- Does the long retrieved passage create distinctive long-range dependency heads, or does the model mostly preserve the same structural patterns as the non-RAG prompt?

Observation notes to fill after inspection:

- TODO: Which representative Layer/Head pairs show visible answer-neighborhood attention?
- TODO: Does the model prefer the retrieved context answer span or the original context answer span?
- TODO: Are the most visually salient structures answer-related, or dominated by sinks/format tokens?
- TODO: Which figure is strongest for the EMNLP demo paper: overview, PCA selection, or a representative detailed heatmap?
